# Phase A — RF-DETR-Nano 5-fold trainer (Colab T4)

**This notebook is the ONLY place CUDA / Colab pins may appear** (the `!pip` cell below). `src/` is MPS/CPU-only.

The training recipe is NOT duplicated here: every knob lives in `configs/detect_rfdetr.yaml` and is consumed through `src.detect.train_rfdetr.train_fold`, so this notebook cannot drift from the repo.

Phase A is the binary spine (pain / no_pain) and the only Phase that emits a *validated* number under the firewall: detector P/R and the fixed-recall operating point are reported on vet-confirmable binary labels, never on VLM-derived AU labels. The RF-DETR backbone is plumbing, never claimed novel.

Operating point = FIXED pain-recall >= 0.90 (Evangelista anchor), set LAST: the highest threshold still meeting the recall floor, chosen inside the train folds and frozen before touching test. Report mean +/- SD across the 5 cat-grouped folds. Augmented copies never enter any reported N.

In [ ]:
# CUDA pins live ONLY in this cell (rfdetr>=1.6.0 adds MPS support + the §2.4 train knobs).
!pip -q install "rfdetr>=1.6.0" supervision scikit-learn torchmetrics pyyaml

In [ ]:
# T4 GPU assert — the ONLY torch.cuda reference in the repo.
import torch
assert torch.cuda.is_available(), "Switch runtime -> T4 GPU"
print("cuda", torch.cuda.get_device_name(0))

## Setup — code + data

`src/` and `configs/` come from the repo clone (single source of truth). The cat-grouped fold dirs `datasets/fold{0..4}` (train/ valid/ test/, each with `_annotations.coco.json`) are built by the §2.2 re-split (`src/data/folds.py`) and copied in from Drive — they are NOT rebuilt here. `CAT_01` is the frozen LOIO hold-out and must not appear in any CV fold.

In [ ]:
# Code: clone the repo so src/ + configs/ are importable
# (private repo -> use a https://<token>@github.com/... URL or gh auth).
import sys
from pathlib import Path

if not Path("cat-fgs-llm").exists():
    !git clone https://github.com/mingrath/cat-fgs-llm.git
%cd cat-fgs-llm
sys.path.insert(0, str(Path.cwd()))

# Data: prebuilt fold dirs from Drive.
from google.colab import drive
drive.mount("/content/drive")

DATA_SRC = "/content/drive/MyDrive/cat-fgs-llm/datasets"  # prebuilt fold0..fold4
if not Path("datasets/fold0").exists():
    !cp -r "{DATA_SRC}" datasets

missing = [k for k in range(5)
           if not Path(f"datasets/fold{k}/train/_annotations.coco.json").exists()]
assert not missing, (
    f"missing fold dirs {missing}: build them with the §2.2 re-split "
    "(src/data/folds.py -> folds.csv -> per-fold COCO export) before training."
)

## num_classes footgun (§2.4)

RF-DETR sizes its classification head from `num_classes`, which must cover the max COCO `category_id` present. Do NOT hard-code 2. Derive the class names from the actual COCO files and assert they are IDENTICAL across all 5 folds before the loop (expect sorted `['no_pain','pain']`).

In [ ]:
import json

def class_names_from_coco(coco_json):
    """Read sorted COCO class names (the footgun-box check) from one split."""
    with open(coco_json) as f:
        cats = json.load(f)["categories"]
    return sorted(c["name"] for c in cats)

per_fold = {
    k: class_names_from_coco(f"datasets/fold{k}/train/_annotations.coco.json")
    for k in range(5)
}
assert len(set(map(tuple, per_fold.values()))) == 1, (
    f"class names differ across folds: {per_fold}"
)
CLASS_NAMES = per_fold[0]
print(CLASS_NAMES)  # expect ['no_pain', 'pain']; num_classes derives from this in train_fold

In [ ]:
# Per-fold training — §2.4 recipe read from configs/detect_rfdetr.yaml via train_fold.
# device is OMITTED inside train_fold so RF-DETR auto-detects CUDA on the T4
# (pass device="mps" only for the M4 smoke run). Run all 5 cat-grouped folds;
# CAT_01 is the frozen LOIO hold-out, never in CV.
from src.detect.train_rfdetr import train_fold

for K in range(5):
    print(f"=== fold {K} ===")
    train_fold(K, class_names=CLASS_NAMES)  # -> ./out/fold{K}

In [ ]:
# Colab VMs are ephemeral — persist checkpoints + tensorboard logs to Drive.
# Re-run this cell after each fold if you expect disconnects mid-loop.
!zip -qr out_rfdetr.zip out
!cp out_rfdetr.zip /content/drive/MyDrive/cat-fgs-llm/
print("saved -> /content/drive/MyDrive/cat-fgs-llm/out_rfdetr.zip")

Evaluate with `src/detect/eval.py`: PR-AUC (Average Precision) is the primary rank metric; the operating point is the highest threshold still meeting pain-recall >= 0.90 inside the train folds, frozen, then applied to the held-out test fold; report mean +/- SD across the 5 cat-grouped folds with bootstrap CIs. `torchmetrics` mAP is localization-only. Never rank on accuracy / ROC-AUC; never write "we beat 77/79/95%".